In [1]:
import pandas as pd

df = pd.read_csv("cleaned_retail_data_updated.csv")

In [2]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TransactionMonth,CohortMonth,CohortIndex
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,01-12-2010 08:26,2.55,17850,United Kingdom,01-12-2010,01-12-2010,0.0
1,536365,71053,WHITE METAL LANTERN,6,01-12-2010 08:26,3.39,17850,United Kingdom,01-12-2010,01-12-2010,0.0
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,01-12-2010 08:26,2.75,17850,United Kingdom,01-12-2010,01-12-2010,0.0
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,01-12-2010 08:26,3.39,17850,United Kingdom,01-12-2010,01-12-2010,0.0
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,01-12-2010 08:26,3.39,17850,United Kingdom,01-12-2010,01-12-2010,0.0


## Revenue per Transaction

Revenue generated from each transaction is calculated by multiplying the quantity of items purchased by their unit price.

Formula:

Revenue = Quantity × UnitPrice

This metric represents the monetary value generated from each transaction and serves as the foundation for calculating Average Order Value (AOV) and Customer Lifetime Value (CLTV).

In [3]:
df['Revenue'] = df['Quantity'] * df['UnitPrice']

In [4]:
df[['InvoiceNo', 'Quantity', 'UnitPrice', 'Revenue']].head()

,InvoiceNo,Quantity,UnitPrice,Revenue
0,536365,6,2.55,15.30
1,536365,6,3.39,20.34
2,536365,8,2.75,22.00
3,536365,6,3.39,20.34
4,536365,6,3.39,20.34


## Average Order Value (AOV) per Cohort

Average Order Value (AOV) measures the average amount spent per order within each customer cohort.

Formula:

AOV = Total Revenue / Number of Orders

Customer cohorts are formed based on the month of their first purchase. AOV helps understand spending behaviour across different groups of customers and identifies cohorts that generate higher revenue per transaction.

In [5]:
order_value = (
    df.groupby(['InvoiceNo', 'CohortMonth'])['Revenue']
      .sum()
      .reset_index()
)

In [6]:
aov = (
    order_value.groupby('CohortMonth')['Revenue']
               .mean()
               .reset_index()
)

aov.columns = ['CohortMonth', 'AOV']

In [7]:
aov.sort_values('CohortMonth')

,CohortMonth,AOV
0,01-01-2011,561.094165
1,01-02-2011,659.819500
2,01-03-2011,412.305050
3,01-04-2011,396.847019
4,01-05-2011,372.279009
5,01-06-2011,460.109465
6,01-07-2011,363.356646
7,01-08-2011,498.740157
8,01-09-2011,433.970019
9,01-10-2011,362.477370


In [8]:
aov.to_csv('aov_per_cohort.csv', index=False)

## Cohort Segmentation by Customer Geography

The dataset does not contain explicit customer acquisition information such as marketing campaigns, referral sources, or acquisition channels. Therefore, customer geography (Country) is used as a proxy segmentation variable.

Revenue is analysed across customer cohorts and countries to understand which geographical segments contribute the highest revenue and Average Order Value (AOV).

In [9]:
country_revenue = (
    df.groupby(['CohortMonth', 'Country'])['Revenue']
      .sum()
      .reset_index()
      .sort_values(['CohortMonth', 'Revenue'],
                   ascending=[True, False])
)

country_revenue.head()

,CohortMonth,Country,Revenue
7,01-01-2011,United Kingdom,489562.76
0,01-01-2011,Australia,126497.13
5,01-01-2011,Spain,23495.91
4,01-01-2011,Germany,16433.90
3,01-01-2011,France,15094.14


In [10]:
order_value_country = (
    df.groupby(['InvoiceNo', 'CohortMonth', 'Country'])['Revenue']
      .sum()
      .reset_index()
)

In [11]:
country_aov = (
    order_value_country
      .groupby(['CohortMonth', 'Country'])['Revenue']
      .mean()
      .reset_index()
)

country_aov.columns = ['CohortMonth', 'Country', 'AOV']

In [12]:
country_revenue.to_csv(
    'cohort_country_revenue.csv',
    index=False
)

country_aov.to_csv(
    'cohort_country_aov.csv',
    index=False
)

The analysis indicates that customer spending behaviour varies across geographical segments. By grouping customers according to their cohort month and country, it becomes possible to identify regions that contribute higher revenue and exhibit greater Average Order Values. These insights can help businesses prioritize high-value markets and design region-specific marketing strategies.